# Dynamic Runtime Context

**Dynamic runtime context** is *mutable* data that evolves during a single run. Unlike static context (which is fixed at the start), dynamic context **changes as the agent works** — tool calls update it, nodes write to it, the LLM reads from it on every step.

In LangGraph this is the **state object** — also called the agent's **short-term memory** for one run.

Typical examples:
- Conversation history (`messages`) — the canonical dynamic field
- A user fact the agent **learned** mid-conversation (e.g. their name)
- Counters, flags, intermediate results

> Source: [LangChain Context docs](https://docs.langchain.com/oss/python/concepts/context#dynamic-runtime-context)

## Setup

Make sure you have a `.env` file in your project directory with:

```
OPENAI_API_KEY=your-api-key-here
OPENAI_ENDPOINT=your-endpoint-here
```

In [32]:
import os
from dotenv import load_dotenv

load_dotenv()

assert os.environ.get("OPENAI_API_KEY"), "OPENAI_API_KEY is not set! Check your .env file."
assert os.environ.get("OPENAI_ENDPOINT"), "OPENAI_ENDPOINT is not set! Check your .env file."

from typing import Annotated
from operator import add
from langchain_openai import ChatOpenAI
from langchain.agents import create_agent, AgentState
from langchain.agents.middleware import dynamic_prompt, ModelRequest
from langchain.tools import tool, ToolRuntime
from langchain_core.messages import HumanMessage, ToolMessage
from langgraph.types import Command

llm = ChatOpenAI(base_url=os.environ["OPENAI_ENDPOINT"], model="gpt-5.4-mini")

print("Environment variables loaded successfully!")

Environment variables loaded successfully!


## Step 1: Look at the default `AgentState`

LangChain provides `AgentState` as the default. It's a `TypedDict` with three fields out of the box.

In [33]:
for name, annotation in AgentState.__annotations__.items():
    print(f"{name}: {annotation}")

messages: ForwardRef('Required[Annotated[list[AnyMessage], add_messages]]', module='langchain.agents.middleware.types')
jump_to: ForwardRef('NotRequired[Annotated[JumpTo | None, EphemeralValue, PrivateStateAttr]]', module='langchain.agents.middleware.types')
structured_response: ForwardRef('NotRequired[Annotated[ResponseT, OmitFromInput]]', module='langchain.agents.middleware.types')


## Step 2: Extend `AgentState` with a custom field

We'll add a `user_name` field. We'll start the run with it **empty** — the agent should fill it in once the user introduces themselves.

In [34]:
class CustomAgentState(AgentState):
    user_name: str

## Step 3: Tools that read and write state

Two pieces:

1. A `dynamic_prompt` that **reads** the current state. It runs before every LLM call, so it always reflects the latest `user_name`.
2. A `remember_name` tool that **writes** to state. Tools update state by returning a `Command(update={...})` — that's the LangGraph mechanism for tool-driven state changes.

Plus a regular `get_weather` tool so the agent has something to do.

In [35]:
@dynamic_prompt
def personalized_prompt(request: ModelRequest) -> str:
    user_name = request.state.get("user_name", "")
    if user_name:
        return f"You are a helpful assistant. The user's name is {user_name}. Address them by name."
    return (
        "You are a helpful assistant. You don't know the user's name yet. "
        "If they introduce themselves, call the `remember_name` tool to save it."
    )


@tool
def remember_name(name: str, runtime: ToolRuntime) -> Command:
    """Save the user's name to state once they've introduced themselves."""
    return Command(update={
        "user_name": name,
        "messages": [ToolMessage(content=f"Saved name: {name}", tool_call_id=runtime.tool_call_id)],
    })


@tool
def get_weather(location: str) -> str:
    """Get the weather for a location."""
    return f"It's sunny and 22°C in {location}."

## Step 4: Register the state schema with the agent

Pass `state_schema=CustomAgentState` and the middleware so the agent knows about the extra field.

In [36]:
agent = create_agent(
    model=llm,
    tools=[remember_name, get_weather],
    state_schema=CustomAgentState,
    middleware=[personalized_prompt],
)

## Step 5: Invoke and watch the state mutate

We start with `user_name=""` (unknown). The user message contains a name — `"Hi, I'm Maria..."`.

Trace what happens:

1. First LLM call: prompt says *"you don't know the user's name"* → the LLM calls `remember_name("Maria")`.
2. The tool returns `Command(update={"user_name": "Maria", ...})` → state mutates.
3. Second LLM call: prompt now says *"the user's name is Maria"* → the LLM addresses her by name and answers her real question.

The same field changed mid-run, driven by a tool. That's dynamic context.

In [37]:
response = agent.invoke({
    "messages": [HumanMessage(content="Hi, I'm Maria! What's the weather in Berlin?")],
    "user_name": "",  # starts empty, will be filled by the tool
})

print(f"Final user_name in state: {response['user_name']!r}\n")

for message in response["messages"]:
    message.pretty_print()

Final user_name in state: 'Maria'

================================ Human Message =================================

Hi, I'm Maria! What's the weather in Berlin?
================================== Ai Message ==================================
Tool Calls:
  remember_name (call_Yp8Z2TgAjhJhSQ0KH2CB0zpr)
 Call ID: call_Yp8Z2TgAjhJhSQ0KH2CB0zpr
  Args:
    name: Maria
================================= Tool Message =================================
Name: remember_name

Saved name: Maria
================================== Ai Message ==================================
Tool Calls:
  get_weather (call_028ykI2UdHC87btyts1CxfAu)
 Call ID: call_028ykI2UdHC87btyts1CxfAu
  Args:
    location: Berlin
================================= Tool Message =================================
Name: get_weather

It's sunny and 22°C in Berlin.
================================== Ai Message ==================================

Hi Maria! It’s sunny and 22°C in Berlin.


## Dynamic context in LangGraph

In a raw **LangGraph** workflow, dynamic context is just the `State` `TypedDict`. Each node receives `state` and returns a dict of updates. Reducers (like `add`) decide how updates are merged.

Below is a tiny two-node workflow that increments a counter twice.

In [38]:
from typing_extensions import TypedDict
from langgraph.graph import StateGraph, START, END


class State(TypedDict):
    counter: Annotated[int, add]  # reducer: new values are added


def increment_node(state: State) -> State:
    print(f"Before: counter={state['counter']}")
    return {"counter": 1}  # add 1 to whatever counter is now


builder = StateGraph(State)
builder.add_node("first", increment_node)
builder.add_node("second", increment_node)
builder.add_edge(START, "first")
builder.add_edge("first", "second")
builder.add_edge("second", END)
graph = builder.compile()

result = graph.invoke({"counter": 0})
print(f"Final counter: {result['counter']}")

Before: counter=0
Before: counter=1
Final counter: 2


Notice each node only said `"counter": 1`, but the reducer added them on top of the existing value — final result is `2`.

## Try it yourself 🛠️

Extend the LangGraph counter workflow above. Add a second field — `log` — that accumulates a string from every node it visits.

Then:


1. Update `increment_node` so it returns **both** `"counter": 1` **and** `"log": [f"visited at counter={state['counter']}"]`.
2. Add a third node `"third"` (you can reuse `increment_node`) and wire it after `"second"`.
3. Invoke the graph with `{"counter": 0, "log": []}` and print the final `counter` and `log`.

You should end up with `counter=3` and a 3-entry log.

In [ ]:
# Your solution here

class State(TypedDict):
    counter: Annotated[int, add]
    log: Annotated[list[str], add]


def increment_node(state: State) -> State:
    # TODO: return both an updated counter AND an entry appended to log
    ...


builder = StateGraph(State)
builder.add_node("first", increment_node)
builder.add_node("second", increment_node)
# TODO: add a "third" node and wire the edges START -> first -> second -> third -> END
builder.add_edge(START, "first")
builder.add_edge("first", "second")
# ...

graph = builder.compile()

result = graph.invoke({"counter": 0, "log": []})
print(f"Final counter: {result['counter']}")
# TODO: print result["log"]